In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from astropy import units as u
from dusty_colors.dust_extinction_fit import DEFAULT_FILTER_WAVELENGTHS_UM
from dust_extinction.averages import G03_SMCBar
from dust_extinction.parameter_averages import F99
from pathlib import Path
from scipy.optimize import least_squares

In [ ]:
ROOT = Path().resolve().parents[0]
STACK_DIR = ROOT / "results" / "stacks" / "dp1_default"
STACK_PATH = STACK_DIR / "stack_fcolors.npz"

data = np.load(STACK_PATH)

fg_redshift = 0.36
waves = {
    band: DEFAULT_FILTER_WAVELENGTHS_UM[band] / (1 + fg_redshift) for band in "griz"
}
x_rest = np.array([waves[band] for band in "giz"])
xr_rest = waves["r"]

rnorm_kpc = 20

In [ ]:
def get_points_for_bins(bins):
    """Return g-r, i-r, and z-r points with jackknife-propagated errors."""
    y = []
    yerr = []
    for i in bins:
        # Get averages
        g_r = float(data["g-r_avg"][i])
        r_i = float(data["r-i_avg"][i])
        i_z = float(data["i-z_avg"][i])
        r_z = r_i + i_z
        y.append(np.array([g_r, -r_i, -r_z]))

        # Get jackknife samples
        g_r_samples = data["g-r_jackknife_samples"][:, i]
        r_i_samples = data["r-i_jackknife_samples"][:, i]
        i_z_samples = data["i-z_jackknife_samples"][:, i]
        r_z_samples = r_i_samples + i_z_samples
        samples = np.column_stack(
            [
                g_r_samples,
                -r_i_samples,
                -r_z_samples,
            ]
        )

        # Calculate jackknife covariance
        centered = samples - samples.mean(axis=0)
        covariance = (1.0 - 1.0 / samples.shape[0]) * centered.T @ centered
        yerr.append(np.sqrt(np.diag(covariance)))

    y = np.array(y)
    yerr = np.array(yerr)

    return y, yerr


def model(x, A, index, law, bins, rnorm_kpc):
    """Predict color excess with respect to the r-band"""
    # Excess as a function of wavelength
    ratio = law((1.0 / x) * u.micron**-1)
    ratio_r = law((1.0 / xr_rest) * u.micron**-1)
    excess = ratio - ratio_r

    # Radial power law normalization
    radii = data["g-r_bin_centers"][bins]
    norm = A * (radii / rnorm_kpc) ** index

    # Rows = radial bins
    # Columns = wavelength bins
    y_model = excess[None, :] * norm[:, None]

    return y_model


def fit_powerlaw(law, bins, rnorm_kpc=20, return_result=False):
    # Get points for each bin
    y, yerr = get_points_for_bins(bins)

    # Function that returns weighted residuals
    def resid(params):
        A, index = params

        # Generate model template
        y_model = model(x_rest, A, index, law, bins, rnorm_kpc=rnorm_kpc)

        return np.ravel((y_model - y) / yerr)

    result = least_squares(resid, [0.1, -1])

    if return_result:
        return result

    if not result.success:
        raise RuntimeError("Optimization failed")

    # Extract results
    out = {
        "model_type": "powerlaw",
        "rnorm_kpc": rnorm_kpc,
        "dust_law": law,
        "param": result.x,
    }

    fisher = result.jac.T @ result.jac
    out["param_cov"] = np.linalg.pinv(fisher)
    out["param_err"] = np.sqrt(np.diag(out["param_cov"]))
    out["snr"] = np.abs(out["param"] / out["param_err"])

    out["chi2"] = np.sum(resid(result.x) ** 2)
    out["dof"] = y.size - len(out["param"])
    out["chi2/dof"] = out["chi2"] / out["dof"]

    return out


def fit_norm(law, bins, return_result=False):
    # Get points for each bin
    y, yerr = get_points_for_bins(bins)

    # Function that returns weighted residuals
    def resid(params):
        A = params

        # Generate model template
        y_model = model(x_rest, A, 0, law, bins, rnorm_kpc=100)

        return np.ravel((y_model - y) / yerr)

    result = least_squares(resid, [y.mean()])

    if return_result:
        return result

    if not result.success:
        raise RuntimeError("Optimization failed")

    # Extract results
    out = {
        "model_type": "norm",
        "dust_law": law,
        "param": result.x,
    }

    fisher = result.jac.T @ result.jac
    out["param_cov"] = np.linalg.pinv(fisher)
    out["param_err"] = np.sqrt(np.diag(out["param_cov"]))
    out["snr"] = np.abs(out["param"] / out["param_err"])

    out["chi2"] = np.sum(resid(result.x) ** 2)
    out["dof"] = y.size - len(out["param"])
    out["chi2/dof"] = out["chi2"] / out["dof"]

    return out


def evaluate_model(x, params, bins):
    if params["model_type"] == "powerlaw":
        A, index = params["param"]
        return model(x, A, index, params["dust_law"], bins, params["rnorm_kpc"])
    else:
        A, index = params["param"], 0.0
        return model(x, A, index, params["dust_law"], bins, 100)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(3, 5), dpi=150, sharex=True)

# Define dust laws we will fit
laws = {
    "MW": {
        "model": F99(Rv=3.1),
        "color": "C0",
        "ytext": 0.6,
    },
    "SMC": {
        "model": G03_SMCBar(),
        "color": "C3",
        "ytext": 0.8,
    },
}

# Loop over radial bins
x_grid = np.linspace(0.3, 0.7, 1000)
y, yerr = get_points_for_bins([0, 1, 2, 3])
for i, (ax, yi, ei) in enumerate(zip(axes, y, yerr)):
    # Plot measurements
    ax.scatter(xr_rest, 0, marker="x", c="k", zorder=10)
    ax.errorbar(x_rest, yi, ei, ls="", marker=".", capsize=2, c="k", zorder=10)

    # Fit and plot models
    for name, law in laws.items():
        # Fit and plot
        result = fit_norm(law["model"], [i])
        y_model = evaluate_model(x_grid, result, [i])
        ax.plot(x_grid, y_model[0], c=law["color"], ls="--")

        # Print chi2/dof
        text = rf"{name}: $\chi^2/\nu$ = {result["chi2/dof"]:.2f}"
        ax.text(
            0.95,
            law["ytext"],
            text,
            transform=ax.transAxes,
            color=law["color"],
            fontsize=7,
            ha="right",
        )

# Axis settings
fig.supylabel(r"Relative Extinction $A_\lambda - A_r$ [mag]", x=-0.07)
axes[-1].set_xlabel(r"$\lambda_\mathrm{rest}$ [$\mu$m]")
for ax in axes:
    ax.set(xlim=(x_grid.min(), x_grid.max()))
    ax.minorticks_on()
    ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)
axes[0].set(ylim=(-0.2, 0.25))
axes[1].set(ylim=(-0.045, 0.065), yticks=np.arange(-0.04, 0.08, 0.04))

# Label bands at top of figure
for band in waves:
    axes[0].text(waves[band], 0.27, f"${band}$", ha="center", va="bottom")

xtext, ytext = 0.03, 0.1
axes[0].text(
    xtext,
    ytext,
    r"$10 < r_\perp < 15 ~ \mathrm{kpc}$",
    transform=axes[0].transAxes,
    fontsize=7,
)
axes[1].text(
    xtext,
    ytext,
    r"$15 < r_\perp < 40 ~ \mathrm{kpc}$",
    transform=axes[1].transAxes,
    fontsize=7,
)
axes[2].text(
    xtext,
    ytext,
    r"$40 < r_\perp < 120 ~ \mathrm{kpc}$",
    transform=axes[2].transAxes,
    fontsize=7,
)
axes[3].text(
    xtext,
    ytext,
    r"$120 < r_\perp < 1000 ~ \mathrm{kpc}$",
    transform=axes[3].transAxes,
    fontsize=7,
)


fig.subplots_adjust(hspace=0.1)

In [ ]:
Av = []
Av_err = []
for i in range(4):
    result = fit_norm(laws["SMC"]["model"], [i])
    Av.append(result["param"][0])
    Av_err.append(result["param_err"][0])

Av = np.array(Av)
Av_err = np.array(Av_err)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dusty_colors.dust_extinction_fit import color_excess_to_av
from astropy.cosmology import Planck18
import astropy.units as u
from scipy.optimize import curve_fit
import itertools


def arcmin_per_kpc(z):
    D_A = Planck18.angular_diameter_distance(z).to(u.kpc).value
    return (180 * 60) / (np.pi * D_A)


def add_angular_axis(ax, z=0.36):
    ax2 = ax.twiny()
    ax2.set_xscale("log")
    ax2.set_xlim(np.array(ax.get_xlim()) * arcmin_per_kpc(z))
    ax2.set_xlabel(r"$\Delta\theta$ [arcmin]")
    return ax2


menard = np.array(
    [
        0.08101611782270275,
        0.027325115693953635,
        0.1445439770745927,
        0.01453876351765176,
        0.2578864782716815,
        0.006006241988800526,
        0.4540909610972468,
        0.005436530402163531,
        0.8101611782270248,
        0.0022745300544756383,
        1.4454397707459257,
        0.001481730535008559,
        2.545155291215893,
        0.0011968443573583598,
        4.5409096109724745,
        0.0007601971142079824,
        8.101611782270272,
        0.0005620161849994557,
        14.265457829751824,
        0.0003900166992555142,
        25.451552912158917,
        0.00016948707226320585,
        0.08101611782270263,
        0.032620084847586425,
        0.1445439770745924,
        0.017577012231528757,
        0.2578864782716818,
        0.007542302116693731,
        0.45409096109724745,
        0.006327845791973369,
        0.8101611782270255,
        0.0028925926886768727,
        1.4265457829751862,
        0.0019082867818400165,
        2.5451552912158957,
        0.0014653783286947037,
        4.540909610972477,
        0.0009915413541835426,
        7.995712002123406,
        0.0007238077583441449,
        14.265457829751854,
        0.0005151839485815968,
        25.45155291215897,
        0.0002672629609220857,
    ]
)


menard = menard.reshape(-1, 2)

menard_x = menard[:11, 0] / arcmin_per_kpc(0.36)
menard_y = menard[:11, 1]
menard_err = menard[11:, 1] - menard_y


kids = np.array(
    [
        0.2886979346427395,
        0.011606199380672444,
        1.3762400222321953,
        0.002819257671635595,
        2.964720402775661,
        0.0017373694608106565,
        6.4968023261201076,
        0.000993812791274001,
        14.10481883937904,
        0.000728674703166439,
        0.28807007566686127,
        0.021059600810189864,
        1.3624207729595528,
        0.004462679391719287,
        2.9592121518820877,
        0.0028901237710978376,
        6.482379077810904,
        0.0018258097654881933,
        14.076697450547655,
        0.0012581443111154686,
    ]
)

kids = kids.reshape(10, 2)

kids_x = kids[:5, 0] / arcmin_per_kpc(0.3)
kids_y = kids[:5, 1]
kids_err = kids[5:, 1] - kids_y


des = np.array(
    [
        15.552397444408108,
        0.08010254140987187,
        27.180543859161116,
        0.023674545170336268,
        48.70520684028501,
        0.005119618878009029,
        87.27555951958644,
        0.002955887936909974,
        156.3903283366551,
        0.0015608676699604353,
        282.58295285711716,
        0.001015127277499201,
        506.36445097493987,
        0.000640830309435162,
        907.361730842972,
        0.0004106082079616616,
        1625.9145147594486,
        0.00020429493735424467,
        2937.8781255892027,
        0.000053607783267907264,
        15.552397444408099,
        0.08758273607753993,
        27.18054385916117,
        0.025885339028130092,
        48.70520684028494,
        0.00612043173446062,
        87.27555951958635,
        0.0033295400976661622,
        156.39032833665493,
        0.002101874571157773,
        280.23807503357045,
        0.0012502049107660286,
        506.36445097493987,
        0.0007892444598565458,
        907.361730842972,
        0.0005529276873581189,
        1625.9145147594486,
        0.00032888399694224,
        2937.8781255892027,
        0.00021073424659847644,
    ]
)

des = des.reshape(20, 2)

des_x = des[:10, 0] / Planck18.h
des_y = des[:10, 1]
des_err = des[10:, 1] - des_y

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3), dpi=200)

dp1_x = data["g-z_bin_centers"]
dp1_y = Av
dp1_err = Av_err

ax.errorbar(
    dp1_x,
    dp1_y,
    dp1_err,
    ls="",
    marker="s",
    markersize=3,
    label="Rubin DP1",
    c="r",
    zorder=100,
)

ax.errorbar(
    menard_x,
    menard_y,
    menard_err,
    ls="",
    marker=".",
    label="Menard 2010",
    mfc="w",
    c="darkgreen",
)

ax.errorbar(
    kids_x,
    kids_y,
    kids_err,
    ls="",
    marker=".",
    label="Genc 2025",
    mfc="w",
    c="gray",
)
ax.errorbar(
    des_x,
    des_y,
    des_err,
    ls="",
    marker=".",
    label="McCleary 2026",
    mfc="w",
    c="darkblue",
)

ax.set(
    xscale="log",
    yscale="log",
    xlabel=r"$r_\perp$ [kpc]",
    ylabel=r"$A_V$ [mag]",
    xlim=(10, 1e4),
    ylim=(1e-5, 1),
)
ax.set_box_aspect(1)
ax.legend(loc="upper right", fontsize=7, frameon=False)

add_angular_axis(ax)

In [ ]:
fit_powerlaw(laws["SMC"]["model"], [0, 1, 2])

In [ ]:
fit_powerlaw(laws["SMC"]["model"], [0, 1])

In [ ]:
fit_powerlaw(laws["SMC"]["model"], [0, 1, 2, 3])

In [ ]:
np.geomspace(10, 500, 5)

In [ ]:
np.geomspace(10, 100, 4)